# Proxy Target Engineering for Credit Risk Modeling

## Objective

The objective of this notebook is to create a target variable that can be used for supervised machine learning models in the credit risk prediction pipeline.

The provided transaction dataset does not contain a direct indicator showing whether a customer defaulted on a loan. Since supervised classification algorithms require a labeled target variable, a proxy for credit risk must be engineered from customer transaction behavior.

This notebook develops that proxy target using the **RFM (Recency, Frequency, Monetary)** framework.

---

## Why a Proxy Target is Needed

In traditional credit scoring systems, historical repayment behavior is used to determine whether a customer is a good or bad borrower.

Examples of direct labels include:

- Loan Default (Yes/No)
- Delinquency Status
- Days Past Due
- Charge-off Status

However, the available Xente transaction dataset contains only customer transaction records and does not include any loan repayment information.

Because of this limitation, a proxy target must be created to represent customer risk behavior.

---

## RFM Framework

RFM analysis is a widely used customer segmentation technique that evaluates customer behavior using three dimensions:

### Recency (R)

Measures how recently a customer made a transaction.

- Lower Recency → More active customer
- Higher Recency → Less active customer

### Frequency (F)

Measures how often a customer transacts.

- Higher Frequency → More engaged customer
- Lower Frequency → Less engaged customer

### Monetary (M)

Measures the total value of customer transactions.

- Higher Monetary Value → Stronger customer activity
- Lower Monetary Value → Weaker customer activity

---

## Assumption for Credit Risk

The underlying business assumption is:

> Customers who transact recently, transact frequently, and generate higher transaction values are more engaged with the platform and are therefore less likely to represent credit risk.

Conversely:

> Customers who have not transacted recently, transact infrequently, and generate low transaction values may represent higher credit risk.

Using this assumption, customers can be grouped into behavioral segments.

---

## Methodology

The proxy target is created using the following steps:

### Step 1: Calculate RFM Metrics

For each customer:

- Recency
- Frequency
- Monetary Value

are calculated from transaction history.

### Step 2: Standardize RFM Features

The three RFM variables are scaled using StandardScaler to ensure equal contribution during clustering.

### Step 3: Customer Segmentation

K-Means clustering is applied to the standardized RFM features.

Customers are grouped into behavioral segments based on similarity in transaction patterns.

### Step 4: Identify High-Risk Segment

The cluster exhibiting:

- High Recency
- Low Frequency
- Low Monetary Value

is considered the least engaged customer segment.

This cluster is labeled as:

`is_high_risk = 1`

All other customers are labeled as:

`is_high_risk = 0`

---

## Expected Output

The final output of this notebook is a customer-level dataset containing:

- RFM Features
- Cluster Assignment
- Binary Risk Label (`is_high_risk`)

This engineered target variable will be used in Task 4 to train and evaluate credit risk classification models.

---

## Business Significance

The generated proxy target allows Bati Bank to:

- Build credit risk models despite the absence of default data.
- Leverage alternative transaction data for underwriting decisions.
- Identify potentially risky customers before extending credit.
- Establish a foundation for future model refinement when real repayment data becomes available.

---

## Notebook Workflow

This notebook follows the following workflow:

1. Import Required Libraries
2. Load Processed Customer Data
3. Compute RFM Metrics
4. Analyze RFM Distributions
5. Standardize RFM Features
6. Apply K-Means Clustering
7. Identify High-Risk Customer Segment
8. Create Binary Target Variable (`is_high_risk`)
9. Validate Target Distribution
10. Save Model-Ready Dataset for Task 4

# Import Libraries

The following libraries are used for:

- Data manipulation and analysis (Pandas, NumPy)
- Data visualization (Matplotlib, Seaborn)
- Feature scaling (StandardScaler)
- Customer segmentation (KMeans)

These tools support the construction of the RFM-based proxy target variable.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load Transaction Dataset

The original transaction dataset is loaded into memory.

Since RFM analysis is performed at the customer level, the raw transaction records will later be aggregated by customer.

In [2]:
df = pd.read_csv("../data/raw/data.csv")

print(df.shape)

df.head()

(95662, 16)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


# 3. Data Preparation for RFM Analysis

Before calculating RFM metrics, the transaction timestamp is converted into datetime format.

A snapshot date is then defined. This date represents the reference point from which customer recency will be calculated.

In [3]:
df["TransactionStartTime"] = pd.to_datetime(
    df["TransactionStartTime"]
)

snapshot_date = (
    df["TransactionStartTime"].max()
    + pd.Timedelta(days=1)
)

The transaction timestamp has been converted successfully and can now be used for time-based feature engineering.
The snapshot date will serve as the reference point for measuring the number of days since each customer's most recent transaction.

# Create RFM Features

RFM stands for:

- Recency: Days since last transaction
- Frequency: Number of transactions
- Monetary: Total transaction value

These metrics summarize customer engagement and purchasing behavior and form the basis of the proxy target construction.

In [4]:
rfm = (
    df.groupby("CustomerId")
    .agg(
        Recency=(
            "TransactionStartTime",
            lambda x:
            (
                snapshot_date - x.max()
            ).days
        ),
        Frequency=(
            "TransactionId",
            "count"
        ),
        Monetary=(
            "Amount",
            "sum"
        )
    )
    .reset_index()
)

rfm.head()

,CustomerId,Recency,Frequency,Monetary
0,CustomerId_1,84,1,-10000.0
1,CustomerId_10,84,1,-10000.0
2,CustomerId_1001,90,5,20000.0
3,CustomerId_1002,26,11,4225.0
4,CustomerId_1003,12,6,20000.0
